### Overview:
This code implements one of the multiple ways of multi-model RAG. It extracts and processes text and images from PDFs, utilizing a multi-modal Retrieval-Augmented Generation (RAG) system for summarizing and retrieving content for question answering.

### Key Components:
   - **PyMuPDF**: For extracting text and images from PDFs.
   - **Gemini 1.5-flash model**: To summarize images and tables.
   - **Cohere Embeddings**: For embedding document splits.
   - **Chroma Vectorstore**: To store and retrieve document embeddings.
   - **LangChain**: To orchestrate the retrieval and generation pipeline.

### Diagram:
   <img src="https://github.com/NirDiamant/RAG_Techniques/blob/main/images/multi_model_rag_with_captioning.svg?raw=1" alt="Reliable-RAG" width="300">

### Motivation:
Efficiently summarize complex documents to facilitate easy retrieval and concise responses for multi-modal data.

### Method Details:
   - Text and images are extracted from the PDF using PyMuPDF.
   - Summarization is performed on extracted images and tables using Gemini.
   - Embeddings are generated via Cohere for storage in Chroma.
   - A similarity-based retriever fetches relevant sections based on the query.

### Benefits:
   - Simplified retrieval from complex, multi-modal documents.
   - Streamlined Q&A process for both text and images.
   - Flexible architecture for expanding to more document types.

### Implementation:
   - Documents are split into chunks with overlap using a text splitter.
   - Summarized text and image content are stored as vectors.
   - Queries are handled by retrieving relevant document segments and generating concise answers.

### Summary:
The project enables multi-modal document processing and retrieval, providing concise, relevant responses by combining state-of-the-art LLMs and vector-based retrieval systems.

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [4]:
# Install required packages
!uv pip install langchain langchain-community pillow pymupdf python-dotenv langchain-cohere

Using Python 3.12.13 environment at: /usr
Resolved 75 packages in 2.26s
Prepared 5 packages in 157ms
Installed 5 packages in 15ms
 + cohere==5.21.1
 + fastavro==1.12.2
 + langchain-cohere==0.5.1
 + types-pyyaml==6.0.12.20260518
 + types-requests==2.33.0.20260518


In [5]:
import fitz  # PyMuPDF
from PIL import Image
import io
import os
from dotenv import load_dotenv

import google.generativeai as genai
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_cohere import ChatCohere, CohereEmbeddings

load_dotenv()

False

### Download the "Attention is all you need" paper

In [6]:
!wget https://arxiv.org/pdf/1706.03762
!mv 1706.03762 attention_is_all_you_need.pdf

--2026-05-19 07:39:47--  https://arxiv.org/pdf/1706.03762
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.67.42, 151.101.131.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2215244 (2.1M) [application/pdf]
Saving to: ‘1706.03762’

1706.03762          100%[===================>]   2.11M  --.-KB/s    in 0.08s   

2026-05-19 07:39:48 (26.4 MB/s) - ‘1706.03762’ saved [2215244/2215244]



### Data Extraction

In [7]:
text_data = []
img_data = []

In [8]:
with fitz.open('attention_is_all_you_need.pdf') as pdf_file:
    # Create a directory to store the images
    if not os.path.exists("extracted_images"):
        os.makedirs("extracted_images")

    # Loop through every page in the PDF
    for page_number in range(len(pdf_file)):
        page = pdf_file[page_number]

        # Get the text on page
        text = page.get_text().strip()
        text_data.append({"response": text, "name": page_number+1})
        # Get the list of images on the page
        images = page.get_images(full=True)

        # Loop through all images found on the page
        for image_index, img in enumerate(images, start=0):
            xref = img[0]  # Get the XREF of the image
            base_image = pdf_file.extract_image(xref)  # Extract the image
            image_bytes = base_image["image"]  # Get the image bytes
            image_ext = base_image["ext"]  # Get the image extension

            # Load the image using PIL and save it
            image = Image.open(io.BytesIO(image_bytes))
            image.save(f"extracted_images/image_{page_number+1}_{image_index+1}.{image_ext}")

In [12]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [15]:
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
model = genai.GenerativeModel(model_name="gemini-2.5-flash")

### Image Captioning

In [16]:
for img in os.listdir("extracted_images"):
    image = Image.open(f"extracted_images/{img}")
    response = model.generate_content([image, "You are an assistant tasked with summarizing tables, images and text for retrieval. \
    These summaries will be embedded and used to retrieve the raw text or table elements \
    Give a concise summary of the table or text that is well optimized for retrieval. Table or text or image:"])
    img_data.append({"response": response.text, "name": img})

In [21]:
img_data[0]

{'response': 'The image illustrates the architecture of a scaled dot-product attention mechanism. It shows inputs Q (Query), K (Key), and V (Value). Q and K are first multiplied (MatMul), then scaled, optionally masked, and processed by a SoftMax function. The output of SoftMax is then multiplied with V (MatMul) to produce the final attention output.',
 'name': 'image_4_1.png'}

In [28]:
text_data[0]

{'response': 'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing wi

### Vectorstore

In [23]:
from google.colab import userdata
COHERE_API_KEY = userdata.get('COHERE_API_KEY')
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

In [24]:
# Set embeddings
embedding_model = CohereEmbeddings(model="embed-english-v3.0")

# Load the document
docs_list = [Document(page_content=text['response'], metadata={"name": text['name']}) for text in text_data]
img_list = [Document(page_content=img['response'], metadata={"name": img['name']}) for img in img_data]

# Split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400, chunk_overlap=50
)

doc_splits = text_splitter.split_documents(docs_list)
img_splits = text_splitter.split_documents(img_list)

In [30]:
!uv pip install chromadb

Using Python 3.12.13 environment at: /usr
Resolved 74 packages in 974ms
Prepared 10 packages in 1.15s
Installed 10 packages in 43ms
 + bcrypt==5.0.0
 + build==1.5.0
 + chromadb==1.5.9
 + durationpy==0.10
 + kubernetes==35.0.0
 + onnxruntime==1.26.0
 + opentelemetry-exporter-otlp-proto-grpc==1.38.0
 + pybase64==1.4.3
 + pypika==0.51.1
 + pyproject-hooks==1.2.0


In [41]:
# Add to vectorstore
vectorstore = Chroma.from_documents(
    documents=doc_splits + img_splits, # adding the both text and image splits
    collection_name="multi_model_rag",
    embedding=embedding_model,
)

retriever = vectorstore.as_retriever(
                search_type="similarity",
                search_kwargs={'k': 1}, # number of documents to retrieve
            )

### Query

In [36]:
query = "What is the BLEU score of the Transformer (base model)?"

In [42]:
docs = retriever.invoke(query)

In [48]:
print(docs[0].page_content)

Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model
BLEU
Training Cost (FLOPs)
EN-DE
EN-FR
EN-DE
EN-FR
ByteNet [18]
23.75
Deep-Att + PosUnk [39]
39.2
1.0 · 1020
GNMT + RL [38]
24.6
39.92
2.3 · 1019
1.4 · 1020
ConvS2S [9]
25.16
40.46
9.6 · 1018
1.5 · 1020
MoE [32]
26.03
40.56
2.0 · 1019
1.2 · 1020
Deep-Att + PosUnk Ensemble [39]
40.4
8.0 · 1020
GNMT + RL Ensemble [38]
26.30
41.16
1.8 · 1020
1.1 · 1021
ConvS2S Ensemble [9]
26.36
41.29
7.7 · 1019
1.2 · 1021
Transformer (base model)
27.3
38.1
3.3 · 1018
Transformer (big)
28.4
41.8
2.3 · 1019
Residual Dropout
We apply dropout [33] to the output of each sub-layer, before it is added to the
sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the


### Output

In [46]:
from langchain_core.output_parsers import StrOutputParser

# Prompt
system = """You are an assistant for question-answering tasks. Answer the question based upon your knowledge.
Use three-to-five sentences maximum and keep the answer concise."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
    ]
)

# LLM
llm = ChatCohere(model="command-r-plus-08-2024", temperature=0)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"documents":docs[0].page_content, "question": query})
print(generation)

The Transformer (base model) achieves a BLEU score of 27.3.


![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--multi-model-rag-with-captioning)